# H&M 고객 데이터 — 전처리(판다스) 4회차 문제


이번 회차는 **머신러닝(분류)** 관점 전처리입니다.  
목표는 `Active`(0/1)를 예측할 수 있도록 모델이 먹을 수 있는 X(피처 행렬)을 만드는 것!

오늘의 핵심 루틴  
- **Split -> Fit -> Transform** (누수 방지)  
- 결측치 + 결측 플래그  
- 희귀 범주 묶기(Rare)  
- 빈도 기반 파생변수(=Frequency Encoding)  
- 원-핫 인코딩 + 컬럼 정렬(reindex)  
- 스케일링(StandardScaler)  

> 데이터가 100만 행이라 노트북이 버벅일 수 있어요.  
> 실습은 아래에서 **샘플링(예: 200,000행)** 후 진행하고, 익숙해지면 샘플 줄을 주석 처리하고 전체로 돌려봅시다.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

df = pd.read_csv("customer_hm.csv")

# 실습 속도 때문에 샘플링 (필요 없으면 주석 처리!)
df = df.sample(200_000, random_state=42).reset_index(drop=True)

print(df.shape)
df.head()


---

### 문제 1 — SPLIT: `Active`를 y로 두고 Train/Test 나누기
- `y = df['Active']`
- `X = df.drop(columns=['Active'])`
- `train_test_split`로 **train/test**를 나누세요.
  - `test_size=0.2`, `random_state=42`, `stratify=y`
- `X_train`, `X_test`, `y_train`, `y_test` 만들고, **각각 shape 출력**


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 2 — 결측 플래그 + 최빈값 채우기: `fashion_news_frequency`
`fashion_news_frequency`에 결측치가 아주 조금 있습니다.

1) 결측 플래그 만들기
- `fashion_news_frequency_isna` 컬럼을 만들고 (결측이면 1, 아니면 0)
- train/test 둘 다에 붙이세요.

2) 결측 채우기 (누수 금지!)
- **train에서만** 최빈값(mode)을 구해서
- train/test에 **동일한 값으로** 결측을 채우세요.

> 힌트: `mode()[0]`


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 3 — 희귀 범주 묶기(Rare): `fashion_news_frequency`
`fashion_news_frequency`는 대부분이 `NONE`/`Regularly`인데, `Monthly`처럼 **아주 작은 범주**가 있습니다.

- train에서 `value_counts()`로 빈도를 구하고
- **빈도 < 1,000** 인 범주는 `Rare`로 묶으세요.
- 같은 규칙을 test에도 적용하세요.

> 기준(1,000 미만)은 **train에서만 정하기!**


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 4 — 빈도 기반 파생변수(Frequency Encoding)
이번에는 `TicketFreq` 같은 느낌으로, 범주 자체를 **빈도 숫자**로 바꿔봅시다.

1) `club_member_status` 빈도
- train에서 `value_counts()`로 빈도표 만들기
- `status_freq` 컬럼을 만들어서 train/test에 붙이기 (`map`)
- test에서 모르는 범주가 나오면 `0`으로 처리 (`fillna(0)`)

2) `fashion_news_frequency` 빈도
- 위와 똑같이 `fn_freq` 컬럼을 만들어서 train/test에 붙이기

> 포인트: `fit(빈도표 만들기)`은 train, `transform(map)`은 train/test 둘 다!


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 5 — 파생변수: 나이 구간 + 뉴스 수신 여부
모델이 패턴을 잡기 쉽게, 아주 가벼운 파생변수를 몇 개 만들어봅시다.

1) `age_band`
- 아래 구간으로 `pd.cut`해서 `age_band`를 만드세요.
  - bins: `[0, 19, 29, 39, 49, 59, 120]`
  - labels: `['10대', '20대', '30대', '40대', '50대', '60대+']`

2) `is_senior`
- 나이가 60 이상이면 1, 아니면 0

3) `news_opt_in`
- `fashion_news_frequency != 'NONE'` 이면 1, 아니면 0

> train/test 둘 다에 만들기!


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 6 — 이상치 캡핑 + 로그 변환(분포 완화)
이번 회차의 포인트 중 하나!

1) `age` IQR 캡핑
- train의 `age`로 Q1/Q3/IQR 계산해서 `lo/hi` 만들기
- `age_cap = age.clip(lo, hi)` 를 train/test에 만들기

2) `log1p` 변환
- `age_log1p = np.log1p(age_cap)` 만들기
- `status_freq_log1p = np.log1p(status_freq)`
- `fn_freq_log1p = np.log1p(fn_freq)`

> 캡핑 경계(lo/hi)도 **train에서만 계산**합니다.


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.
# q1, q3 = X_train["age"].quantile([0.25, 0.75])
# iqr = q3 - q1
# lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr


---

### 문제 7 — 원-핫 인코딩(get_dummies) + 컬럼 정렬(reindex)
이제 모델이 먹을 수 있게 범주형을 원-핫 인코딩합니다.

- 원-핫 대상: `club_member_status`, `fashion_news_frequency`, `age_band`
- `pd.get_dummies(..., drop_first=True)`로 train/test 각각 변환
- **가장 중요한 것!** train 컬럼 기준으로 test 컬럼 정렬:

```python
X_test_dum = X_test_dum.reindex(columns=X_train_dum.columns, fill_value=0)
```

- 변환 후 `shape` 확인


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.
# "customer_id", "age", "age_cap", "status_freq", "fn_freq" 해당 컬럼은 모델 입력에서 빼자! drop!



---

### 문제 8 — 스케일링(StandardScaler): train만 fit!
스케일링이 필요한 수치형 컬럼만 골라서 표준화해봅시다.

- 추천 스케일링 대상
  - `FN`
  - `age_log1p`
  - `status_freq_log1p`
  - `fn_freq_log1p`

- `StandardScaler`를 준비하고
  - train: `fit_transform`
  - test: `transform`

> 결과가 맞으면 train의 평균이 0 근처로 나옵니다(완전 0은 아니어도 OK).


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.


---

### 문제 9 — (맛보기) 로지스틱 회귀로 학습/평가
전처리 결과가 진짜 모델 입력(X)으로 쓸 수 있는지 확인만 해봅시다.

- `LogisticRegression(max_iter=1000)`
- 정확도(accuracy) + ROC-AUC 둘 다 출력

> 데이터가 크면 학습이 부담될 수 있으니, 필요하면 위에서 샘플 수를 더 줄여도 됩니다.


In [ ]:
# TODO: 여기에서 풀이 코드를 작성하세요.
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import accuracy_score, roc_auc_score

# model = LogisticRegression(max_iter=1000)
# model.fit(X_train_dum, y_train)

# pred = model.predict(X_test_dum)
# proba = model.predict_proba(X_test_dum)[:, 1]

# print("Accuracy:", accuracy_score(y_test, pred))
# print("ROC-AUC :", roc_auc_score(y_test, proba))